# Stage 6 Screening Analysis & Reporting
Formal Hybrid Stage 6 only. This notebook never runs evaluation, selection, or OOS.

## 00 Parameters and Source Validation

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import display
from factor_gfn.reporting import load_stage6_report_data, Stage6ReportRenderer

HYBRID_RUN_ID = 'hybrid_5_15_k16_seed42_20260816T025559Z'
STAGE6_ROOT = (
    REPO_ROOT / 'runs' / 'stage6' / 'hybrid_provisional' / HYBRID_RUN_ID
)

def unique_manifest(relative_pattern, label):
    matches = sorted(STAGE6_ROOT.glob(relative_pattern))
    if len(matches) != 1:
        raise RuntimeError(
            f'{label} 应恰好存在 1 个，实际为 {len(matches)} 个：{matches}。'
            '请先完整运行 run_stage6_hybrid_formal_selection.ipynb，'
            '且不要把其他 Stage 6 run 放入当前正式 run 目录。'
        )
    return matches[0].resolve()

TRAIN_REUSE_MANIFEST = unique_manifest(
    'train_reuse/*/train_reuse_manifest.json', 'Train-reuse manifest'
)
train_reuse = json.loads(TRAIN_REUSE_MANIFEST.read_text(encoding='utf-8'))
SOURCE_SET_MANIFEST = (
    STAGE6_ROOT / 'source_snapshots' / 'source_sets'
    / train_reuse['source_set_fingerprint'] / 'source_set_manifest.json'
).resolve()
CANDIDATE_IMPORT_MANIFEST = (
    STAGE6_ROOT / 'candidate_import' / train_reuse['candidate_registry_fingerprint']
    / 'candidate_import_manifest.json'
).resolve()
COMPATIBILITY_MANIFEST = (
    STAGE6_ROOT / 'compatibility' / train_reuse['compatibility_audit_fingerprint']
    / 'expression_compatibility_manifest.json'
).resolve()
TRAIN_ENTRY_MANIFEST = (
    STAGE6_ROOT / 'train_preparation' / 'train_preparation_entry_manifest.json'
).resolve()
TRAIN_PASS_MANIFEST = (
    STAGE6_ROOT / 'train_preparation' / 'train_pass_manifest'
    / 'train_pass_manifest.json'
).resolve()
VALIDATION_ENTRY_MANIFEST = (
    STAGE6_ROOT / 'validation_evaluation' / 'validation_evaluation_entry_manifest.json'
).resolve()
SELECTION_MANIFEST = unique_manifest(
    'provisional_selection/*/enriched_selection_manifest.json', 'Selection manifest'
)
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'stage6_reporting_v2'

print('Formal Hybrid Stage 6 source:', HYBRID_RUN_ID)
for name, path in {
    'source_set': SOURCE_SET_MANIFEST,
    'candidate_import': CANDIDATE_IMPORT_MANIFEST,
    'compatibility': COMPATIBILITY_MANIFEST,
    'train_entry': TRAIN_ENTRY_MANIFEST,
    'train_pass': TRAIN_PASS_MANIFEST,
    'validation_entry': VALIDATION_ENTRY_MANIFEST,
    'selection': SELECTION_MANIFEST,
}.items():
    print(f'{name}: {path}')

bundle = load_stage6_report_data(
    source_set_manifest_path=SOURCE_SET_MANIFEST,
    candidate_import_manifest_path=CANDIDATE_IMPORT_MANIFEST,
    compatibility_manifest_path=COMPATIBILITY_MANIFEST,
    train_entry_manifest_path=TRAIN_ENTRY_MANIFEST,
    train_pass_manifest_path=TRAIN_PASS_MANIFEST,
    validation_entry_manifest_path=VALIDATION_ENTRY_MANIFEST,
    selection_manifest_path=SELECTION_MANIFEST,
)
renderer = Stage6ReportRenderer(bundle, OUTPUT_DIR)
display(bundle.snapshot_manifest)

## 01 Screening Funnel

In [ ]:
display(bundle.funnel_summary)
renderer.export_table('funnel_summary')
renderer.figure_candidate_screening_funnel(save=True)

## 02 Hard Filter Diagnostics

In [ ]:
display(bundle.hard_filter_condition_summary)
renderer.export_table('hard_filter_condition_summary')
renderer.figure_hard_filter_failure_counts(save=True)

In [ ]:
display(bundle.failure_combinations)
renderer.export_table('failure_combinations')

## 03 Train → Validation Stability

In [ ]:
renderer.figure_train_validation_ic_scatter(save=True)

In [ ]:
renderer.figure_abs_train_validation_ic_scatter(save=True)

In [ ]:
renderer.figure_train_validation_long_ir_scatter(save=True)

In [ ]:
display(bundle.stability_summary)
renderer.export_table('stability_summary')
renderer.export_table('validation_candidate_metrics')

## 04 Quality Before / After

In [ ]:
renderer.figure_ic_distribution_before_after(save=True)

In [ ]:
renderer.figure_long_ir_distribution_before_after(save=True)

In [ ]:
display(bundle.before_after_quality_summary)
renderer.export_table('before_after_quality_summary')
renderer.figure_barra_corr_before_after(save=True)

## 05 Decorrelation

In [ ]:
renderer.figure_train_long_excess_corr_before_top30(save=True)

In [ ]:
renderer.figure_train_long_excess_corr_after_top20(save=True)

In [ ]:
display(bundle.decorrelation_pair_summary)
renderer.export_table('decorrelation_pair_summary')
renderer.export_table('greedy_pair_audit')
renderer.figure_greedy_decorrelation_decisions(save=True)

## 06 Provisional Factor Pool and Frozen-order Top100

In [ ]:
display(bundle.provisional_factor_pool)
renderer.export_table('provisional_factor_pool')

In [ ]:
renderer.figure_provisional_pool_train_validation_ic(save=True)

In [ ]:
renderer.figure_provisional_pool_quality_summary(save=True)
display(bundle.top100_quality_summary)
renderer.export_table('top100_candidate_metrics')
renderer.export_table('top100_quality_summary')
renderer.figure_top100_quality_summary(save=True)
renderer.figure_top100_ic_distribution(save=True)
renderer.figure_top100_long_ir_distribution(save=True)
renderer.figure_top100_barra_corr_distribution(save=True)
display(bundle.complexity_summary)
renderer.export_table('complexity_summary')
renderer.figure_complexity_summary(save=True)

## 07 Ranked Candidate Examples and Expression Structure Shift

In [ ]:
display(bundle.top_candidate_examples)
renderer.export_table('top_candidate_examples')
renderer.figure_top_candidate_examples(save=True)
display(bundle.structure_shift_summary)
renderer.export_table('structure_shift_summary')
renderer.figure_complexity_shift(save=True)

In [ ]:
renderer.export_table('operator_prevalence_shift')
renderer.export_table('field_prevalence_shift')
renderer.export_table('window_prevalence_shift')
renderer.figure_operator_field_preference_shift(save=True)

## 08 Export

In [ ]:
outputs = renderer.render_all()
display(outputs['manifest'])